C:\Users\frank\Desktop\MVP krs kommune\
│
├── .env                       # API-nøkler (GEMINI_API_KEY, SERPER_API_KEY)
├── .gitignore                 # Filer som ikke skal pushes til Git (.env, db, __pycache__)
├── requirements.txt           # Liste over Python-pakker (streamlit, pandas, pydantic, pypdf, osv.)
├── README.md                  # Kort beskrivelse av prosjektet og hvordan det kjøres
│
├── app/                       # Hovedkildekode for applikasjonen
│   ├── __init__.py
│   ├── schemas.py             # Pydantic-skjemaer (datakontrakter)
│   ├── tools.py               # Verktøy (SQL-oppslag, RAG-søk, Serper)
│   ├── agent.py               # LLM-orkestrering og agent-løkke
│   ├── build_rag_index.py     # Indekserings- og CSV-importskript
│   └── frontend.py            # Streamlit chat-grensesnitt
│
├── data/                      # Alt av datagrunnlag
│   ├── db/
│   │   └── kommune_golden_record.db   # SQLite-databasen
│   └── knowledge_base/        # PDF-er, CSV-er og tekstdokumenter (rettet skrivefeil)
│       ├── 1.-tertialrapport-2026---kristiansand-kommune.pdf
│       ├── 2.-tertialrapport-2026---kristiansand-kommune---sterkere-sammen.pdf
│       └── ...
│
├── notebooks/                 # Eksperimentering og analyse
│   └── py_script.ipynb
│
└── assets/                    # Bilder, guider og statiske filer
    ├── FT-Visual Guide.pdf
    └── ft-Viz-Guide.png

In [ ]:
# app/frontend.py
import os
import sys
import streamlit as st

# Sørg for at Python finner modulene i app/-mappen uavhengig av hvorfra Streamlit kjøres
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))

try:
    from app.agent import run_krs_agent
except ImportError:
    from agent import run_krs_agent

# 1. Konfigurer Streamlit-siden
st.set_page_config(
    page_title="Kristiansand Kommune - AI Økonomirådgiver",
    page_icon="🏛️",
    layout="wide"
)

# 2. Tittel og ingress
st.title("🏛️ Kristiansand Kommune — AI Økonomirådgiver (MVP)")
st.caption("Hybrid AI-beslutningsstøtte basert på ERP-transaksjoner (SQL), interne dokumenter (RAG) og eksternt nettsøk (Serper).")

# 3. Sidebar for status og innstillinger
with st.sidebar:
    st.header("⚙️ Konfigurasjon & Status")
    
    gemini_key = os.getenv("GEMINI_API_KEY")
    serper_key = os.getenv("SERPER_API_KEY")
    
    st.write("**API-nøkler i miljøet:**")
    st.write(f"- Gemini API: {'✅ Konfigurert' if gemini_key else '❌ Mangler'}")
    st.write(f"- Serper API: {'✅ Konfigurert' if serper_key else '❌ Mangler'}")
    
    st.divider()
    st.write("**Aktive datakilder:**")
    st.markdown("""
    - 📊 `data/db/kommune_golden_record.db` (SQLite)
    - 📄 `data/knowledge_base/` (PDF/RAG-indeks)
    - 🌐 Serper.dev API (Nettsøk)
    """)
    
    st.divider()
    if st.button("🗑️ Tøm samtalehistorikk"):
        st.session_state.messages = []
        st.rerun()

# 4. Initialiser samtalehistorikk i session state
if "messages" not in st.session_state:
    st.session_state.messages = [
        {
            "role": "assistant",
            "content": "Hei! Jeg er din AI-økonomirådgiver for Kristiansand kommune. Hva kan jeg hjelpe deg med i dag?"
        }
    ]

# 5. Vis tidligere meldinger i chatten
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# 6. Håndter nye spørsmål fra brukeren
if user_input := st.chat_input("Skriv spørsmålet ditt her (f.eks. 'Hva er driftsutgiftene for BOMILJO i 2024?')..."):
    # Legg til brukermelding i chat-grensesnittet
    st.session_state.messages.append({"role": "user", "content": user_input})
    with st.chat_message("user"):
        st.markdown(user_input)

    # Generer og vis svar fra AI-agenten
    with st.chat_message("assistant"):
        with st.spinner("Analyser spørringen, søker i SQL-databasen og dokumentene..."):
            try:
                response = run_krs_agent(user_input)
                st.markdown(response)
                st.session_state.messages.append({"role": "assistant", "content": response})
            except Exception as e:
                error_msg = f"Det oppstod en feil: {str(e)}"
                st.error(error_msg)
                st.session_state.messages.append({"role": "assistant", "content": error_msg})

In [1]:
# inspect_db.py
import sqlite3
import pandas as pd

DB_PATH = "kommune_golden_record.db"

def inspect_database():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    # Hent alle tabeller i databasen
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [row[0] for row in cursor.fetchall()]
    
    print(f"=== Fant {len(tables)} tabell(er) i {DB_PATH} ===\n")
    
    for table in tables:
        print("=" * 50)
        print(f"TABELL: {table}")
        print("=" * 50)
        
        # Hent struktur og kolonnedefinisjoner
        cursor.execute(f"PRAGMA table_info('{table}');")
        columns = cursor.fetchall()
        print("Kolonner og datatyper:")
        for col in columns:
            # col: (col_id, name, type, notnull, dflt_value, pk)
            pk_str = " (PRIMARY KEY)" if col[5] else ""
            print(f"  • {col[1]}: {col[2]}{pk_str}")
            
        # Hent et lite utdrag av dataene
        try:
            df_sample = pd.read_sql_query(f"SELECT * FROM '{table}' LIMIT 3;", conn)
            print("\nEksempeldata (første 3 rader):")
            print(df_sample.to_string(index=False))
        except Exception as e:
            print(f"Kunne ikke hente eksempeldata: {e}")
            
        print("\n")
        
    conn.close()

if __name__ == "__main__":
    inspect_database()

=== Fant 20 tabell(er) i kommune_golden_record.db ===

TABELL: stg_erp_transactions
Kolonner og datatyper:
  • raw_id: INTEGER (PRIMARY KEY)
  • source_system: TEXT
  • voucher_number: TEXT
  • accounting_date: TEXT
  • account_code: TEXT
  • function_code: TEXT
  • cost_center_code: TEXT
  • amount: REAL
  • description: TEXT
  • scale_factor: REAL
  • ingested_at: TIMESTAMP

Eksempeldata (første 3 rader):
 raw_id source_system voucher_number accounting_date account_code function_code cost_center_code    amount         description  scale_factor         ingested_at
      1   BOMILJO_ERP         101000         2024-01       101000           254          BOMILJO 659882.34 Fastlønn - ordinære           1.0 2026-10-16 12:25:19
      2   BOMILJO_ERP         101000         2024-02       101000           254          BOMILJO 956490.83 Fastlønn - ordinære           1.0 2026-10-16 12:25:19
      3   BOMILJO_ERP         101000         2024-03       101000           254          BOMILJO 923021.33

In [16]:
# schemas.py
import importlib
import schemas

# Tvinger Python til å laste oppdaterte schemas.py fra disk
importlib.reload(schemas)

from schemas import ERPSQLQueryInput, KnowledgeSearchInput
from pydantic import BaseModel, Field

class ERPSQLQueryInput(BaseModel):
    query: str = Field(
        ..., 
        description=(
            "Leseorientert SQL-spørring (KUN SELECT) mot tabellen 'stg_erp_transactions'. "
            "Tilgjengelige kolonner: raw_id, source_system, voucher_number, accounting_date (YYYY-MM), "
            "account_code (artskonto), function_code (KOSTRA-funksjon), cost_center_code (ansvar), "
            "amount, description."
        )
    )

class KnowledgeSearchInput(BaseModel):
    query: str = Field(..., description="Søkefrase for tertialrapporter, økonomiplaner og reglementer.")
    max_results: int = Field(default=3, ge=1, le=5)
class SerperSearchInput(BaseModel):
    query: str = Field(..., description="Søkefrase for nettsøk (f.eks. oppdaterte veiledere fra Regjeringen, ROBEK-regler, KS-retningslinjer).")

In [20]:
import os
import requests

def search_web_serper(args: SerperSearchInput) -> str:
    """Utfører nettsøk via Serper.dev API for ekstern informasjon."""

    api_key = os.getenv("SERPER_API_KEY")
    if not api_key:
        return "Feil: SERPER_API_KEY er ikke konfigurert i miljøvariablene."

    endpoint = "https://google.serper.dev/search"

    headers = {
        "X-API-KEY": api_key,
        "Content-Type": "application/json"
    }

    payload = {
        "q": args.query,
        "gl": "no",
        "hl": "no"
    }

    try:
        response = requests.post(
            endpoint,
            headers=headers,
            json=payload,
            timeout=10
        )

        response.raise_for_status()

        data = response.json()

        organic = data.get("organic", [])

        if not organic:
            return f"Ingen relevante treff på nettsøk for '{args.query}'."

        results = []

        for item in organic[:3]:
            results.append(
                f"Tittel: {item.get('title', 'Ingen tittel')}\n"
                f"Lenke: {item.get('link', '')}\n"
                f"Utdrag: {item.get('snippet', '')}"
            )

        return "\n\n".join(results)

    except requests.exceptions.Timeout:
        return "Feil: Nettsøket brukte for lang tid."

    except requests.exceptions.RequestException as e:
        return f"Feil ved HTTP-kall: {e}"

    except ValueError as e:
        return f"Feil ved behandling av JSON-respons: {e}"

    except Exception as e:
        return f"Uventet feil ved nettsøk: {e}"

Pydantic er et av de mest brukte Python-bibliotekene for datavalidering og typehåndtering, basert på ordinære Python type-tilvisninger (type hints).Istedenfor at du må skrive lange if/else-sjekker manuelt for å kontrollere om verdier er av riktig type eller innenfor lovlige rammer, definerer du en Pydantic-modell (BaseModel). Pydantic sørger deretter automatisk for at data som passerer gjennom systemet blir sjekket, konvertert og validert i sanntid.Hvorfor Pydantic er helt sentralt i AI-teknologi og API-erStrenge datakontrakter (Schemas) Du definerer nøyaktig hvordan data skal se ut. For eksempel kan du kreve at en tekststreng ikke kan være tom, at et tall må ligge mellom $1$ og $5$, eller at en ID må følge et bestemt mønster (f.eks. regex ^DOC-\d{3,6}$).Verktøykall (Tool Calling) og AI-agenter Når du ber en kunstig intelligens (som OpenAI eller Gemini) om å kalle et verktøy (f.eks. koble til databasen din kommune_golden_record.db), genererer Pydantic automatisk et JSON Schema fra Python-koden din. AI-modellen bruker dette skjemaet for å forstå nøyaktig hvilke parametere den må levere.Feilhåndtering og selvkorrigering Dersom AI-modellen eller en bruker sender inn feilaktige parametere (f.eks. en tekststreng der det forventes et tall), kaster Pydantic en ValidationError. I en agent-løkke fanger du opp denne feilen og sender feilmeldingen tilbake til AI-modellen slik at den kan korrigere seg selv automatisk.Sømløs integrasjon med FastAPI Pydantic danner selve fundamentet for backend-rammeverket FastAPI. Alt som sendes inn eller ut av REST API-endepunktene dine blir automatisk validert mot Pydantic-modellene dine.Eksempel fra Kristiansand MVP-en vårSlik definerte vi Pydantic-skjemaene for verktøyene i prosjektet ditt:from pydantic import BaseModel, Field

# Definerer en streng datakontrakt for SQL-verktøyet
class ERPSQLQueryInput(BaseModel):
    query: str = Field(
        ..., 
        description="Leseorientert SQL-spørring (KUN SELECT) mot tabellen 'stg_erp_transactions'."
    )

# Definerer en datakontrakt med grenseverdier (ge=1, le=5)
class KnowledgeSearchInput(BaseModel):
    query: str = Field(..., description="Søkefrase for tertialrapporter og økonomiplaner.")
    max_results: int = Field(default=3, ge=1, le=5) # Må være mellom 1 og 5
Når du forsøker å opprette KnowledgeSearchInput(query="test", max_results=10), vil Pydantic umiddelbart stoppe kjøringen og gi en klar feilmelding om at 10 overskrider maksgrensen på 5.

In [2]:
# test_erp_tool.py
from schemas import ERPSQLQueryInput
from tools import query_erp_transactions

def test_erp_query():
    print("=" * 60)
    print("TEST 1: Gyldig SELECT-spørring mot BOMILJO")
    print("=" * 60)
    
    # 1. Opprett Pydantic-objekt med en fiktert test-spørring
    test_query = ERPSQLQueryInput(
        query=(
            "SELECT accounting_date, account_code, function_code, "
            "cost_center_code, amount, description "
            "FROM stg_erp_transactions "
            "WHERE cost_center_code = 'BOMILJO' "
            "LIMIT 3;"
        )
    )
    
    # 2. Kjør verktøyet
    resultat = query_erp_transactions(test_query)
    print("Svar fra databasen (JSON-format for LLM):")
    print(resultat)
    
    print("\n" + "=" * 60)
    print("TEST 2: Sikkerhetskontroll (Forsøk på ulovlig DELETE)")
    print("=" * 60)
    
    # Sjekk at sikkerhetsventilen stopper endringsoperasjoner
    invalid_query = ERPSQLQueryInput(
        query="DELETE FROM stg_erp_transactions WHERE raw_id = 1;"
    )
    sikkerhets_svar = query_erp_transactions(invalid_query)
    print("Respons fra verktøyet:")
    print(sikkerhets_svar)

if __name__ == "__main__":
    test_erp_query()

TEST 1: Gyldig SELECT-spørring mot BOMILJO
Svar fra databasen (JSON-format for LLM):
[{"accounting_date":"2024-01","account_code":"101000","function_code":"254","cost_center_code":"BOMILJO","amount":659882.34,"description":"Fastlønn - ordinære"},{"accounting_date":"2024-02","account_code":"101000","function_code":"254","cost_center_code":"BOMILJO","amount":956490.83,"description":"Fastlønn - ordinære"},{"accounting_date":"2024-03","account_code":"101000","function_code":"254","cost_center_code":"BOMILJO","amount":923021.33,"description":"Fastlønn - ordinære"}]

TEST 2: Sikkerhetskontroll (Forsøk på ulovlig DELETE)
Respons fra verktøyet:
Sikkerhetsfeil: Kun SELECT-spørringer er tillatt.


In [4]:
def import_csv_files(conn):
    """Finner alle CSV-filer og konverterer dem til strukturerte SQL-tabeller."""
    print("=== Skanner etter CSV-filer for tabellimport ===")
    
    search_dirs = [".", KNOWLEDGE_BASE_DIR]
    imported_tables = []
    
    for search_dir in search_dirs:
        if not os.path.exists(search_dir):
            continue
        for file in os.listdir(search_dir):
            if file.lower().endswith(".csv"):
                file_path = os.path.join(search_dir, file)
                raw_name = os.path.splitext(file)[0].lower()
                table_name = f"stg_{raw_name}".replace(" ", "_").replace("-", "_")
                
                # Prøv fleksibel innlesing med støtte for både semikolon og komma
                df = None
                for sep_option in [None, ";", ","]:
                    try:
                        if sep_option is None:
                            df = pd.read_csv(file_path, sep=None, engine="python", encoding="utf-8")
                        else:
                            df = pd.read_csv(file_path, sep=sep_option, encoding="latin-1")
                        break
                    except Exception:
                        continue
                
                if df is not None:
                    df.to_sql(table_name, conn, if_exists="replace", index=False)
                    print(f"  ✓ Importerte CSV '{file}' -> Tabell '{table_name}' ({len(df)} rader)")
                    imported_tables.append(table_name)
                else:
                    print(f"  ✕ Kunne ikke lese CSV '{file}' med standard skilletegn.")
                    
    return imported_tables

In [24]:
# tools.py

import os
import requests

from schemas import (
    ERPSQLQueryInput,
    KnowledgeSearchInput,
    SerperSearchInput,
)


# ============================================================
# 1. ERP / SQL
# ============================================================

# NB:
# Din eksisterende funksjon query_erp_transactions(...)
# skal beholdes her.
#
# Eksempel på forventet signatur:
#
# def query_erp_transactions(args: ERPSQLQueryInput) -> str:
#     ...
#
# Ikke slett den eksisterende implementasjonen din.


# ============================================================
# 2. KNOWLEDGE BASE / RAG
# ============================================================

# NB:
# Din eksisterende funksjon search_knowledge_base(...)
# skal beholdes her.
#
# Eksempel på forventet signatur:
#
# def search_knowledge_base(args: KnowledgeSearchInput) -> str:
#     ...
#
# Ikke slett den eksisterende implementasjonen din.


# ============================================================
# 3. WEB SEARCH / SERPER
# ============================================================

def search_web_serper(args: SerperSearchInput) -> str:
    """
    Utfører nettsøk via Serper.dev API for ekstern informasjon.
    """

    api_key = os.getenv("SERPER_API_KEY")

    if not api_key:
        return (
            "Feil: SERPER_API_KEY er ikke konfigurert "
            "i miljøvariablene."
        )

    endpoint = "https://google.serper.dev/search"

    headers = {
        "X-API-KEY": api_key,
        "Content-Type": "application/json",
    }

    payload = {
        "q": args.query,
        "gl": "no",
        "hl": "no",
    }

    try:
        response = requests.post(
            endpoint,
            headers=headers,
            json=payload,
            timeout=10,
        )

        response.raise_for_status()

        data = response.json()

        organic = data.get("organic", [])

        if not organic:
            return (
                f"Ingen relevante treff på nettsøk "
                f"for '{args.query}'."
            )

        results = []

        for item in organic[:3]:
            results.append(
                f"Tittel: {item.get('title', 'Ingen tittel')}\n"
                f"Lenke: {item.get('link', '')}\n"
                f"Utdrag: {item.get('snippet', '')}"
            )

        return "\n\n".join(results)

    except requests.exceptions.Timeout:
        return "Feil: Nettsøket brukte for lang tid."

    except requests.exceptions.RequestException as e:
        return f"Feil ved HTTP-kall: {e}"

    except ValueError as e:
        return f"Feil ved behandling av JSON-respons: {e}"

    except Exception as e:
        return f"Uventet feil ved nettsøk: {e}"

In [33]:
# agent.py

import json
import os

from openai import OpenAI

from schemas import (
    ERPSQLQueryInput,
    KnowledgeSearchInput,
    SerperSearchInput,
)

from tools import (
    query_erp_transactions,
    search_knowledge_base,
    search_web_serper,
)


# ============================================================
# LEVERANDØRKONFIGURASJON
# ============================================================

USE_OLLAMA = False
# True  = kjør lokalt med Ollama
# False = bruk Gemini via OpenAI-kompatibelt API


if USE_OLLAMA:

    client = OpenAI(
        base_url="http://localhost:11434/v1",
        api_key="ollama",
    )

    MODEL_NAME = os.getenv(
        "OLLAMA_MODEL",
        "qwen2.5",
    )

else:

    gemini_api_key = os.getenv("GEMINI_API_KEY")

    if not gemini_api_key:
        raise ValueError(
            "GEMINI_API_KEY er ikke konfigurert "
            "i miljøvariablene."
        )

    client = OpenAI(
        api_key=gemini_api_key,
        base_url=(
            "https://generativelanguage.googleapis.com/"
            "v1beta/openai/"
        ),
    )

    MODEL_NAME = os.getenv(
        "GEMINI_MODEL",
        "gemini-3.8-flash",
    )


# ============================================================
# VERKTØYDEFINISJONER
# ============================================================

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "query_erp_transactions",
            "description": (
                "Utfører en skrivebeskyttet SQL SELECT-spørring "
                "mot SQLite-databasen."
            ),
            "parameters": ERPSQLQueryInput.model_json_schema(),
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_knowledge_base",
            "description": (
                "Søker i lokale dokumenter som tertialrapporter, "
                "økonomiplaner og forskrifter."
            ),
            "parameters": KnowledgeSearchInput.model_json_schema(),
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_web_serper",
            "description": (
                "Utfører nettsøk via Serper API "
                "for ekstern informasjon."
            ),
            "parameters": SerperSearchInput.model_json_schema(),
        },
    },
]


# ============================================================
# KJØR VERKTØY
# ============================================================

def execute_tool(func_name: str, args: dict) -> str:
    """
    Kjører riktig lokalt verktøy basert på funksjonsnavnet
    modellen returnerer.
    """

    try:

        if func_name == "query_erp_transactions":

            validated_args = ERPSQLQueryInput(**args)

            return str(
                query_erp_transactions(validated_args)
            )

        elif func_name == "search_knowledge_base":

            validated_args = KnowledgeSearchInput(**args)

            return str(
                search_knowledge_base(validated_args)
            )

        elif func_name == "search_web_serper":

            validated_args = SerperSearchInput(**args)

            return str(
                search_web_serper(validated_args)
            )

        else:

            return (
                f"Feil: Ukjent verktøy "
                f"'{func_name}'."
            )

    except Exception as exc:

        return (
            f"Feil under kjøring av verktøyet "
            f"'{func_name}': "
            f"{type(exc).__name__}: {exc}"
        )


# ============================================================
# AGENT
# ============================================================

def run_krs_agent(
    user_prompt: str,
    max_iterations: int = 5,
) -> str:

    """
    Kjører KRS-agenten med støtte for sekvensielle
    og flere verktøykall.
    """

    if not user_prompt or not user_prompt.strip():

        return (
            "Feil: Brukerforespørselen "
            "kan ikke være tom."
        )

    # --------------------------------------------------------
    # Meldingshistorikk
    # --------------------------------------------------------

    messages = [
        {
            "role": "system",
            "content": (
                "Du er en AI-rådgiver for økonomistyring "
                "i Kristiansand kommune.\n\n"

                "Du har tilgang til tre verktøy:\n\n"

                "1. query_erp_transactions\n"
                "Bruk dette for SQL SELECT-oppslag i "
                "SQLite-tabellene:\n"
                "- stg_erp_transactions\n"
                "- stg_kostra_stats_krs_km\n"
                "- stg_kostra_kontoplan\n\n"

                "2. search_knowledge_base\n"
                "Bruk dette for søk i lokale dokumenter, "
                "tertialrapporter, økonomiplaner "
                "og forskrifter.\n\n"

                "3. search_web_serper\n"
                "Bruk dette for nettsøk etter ekstern "
                "og oppdatert informasjon som statlige "
                "veiledere, lover og forskrifter.\n\n"

                "REGLER:\n"
                "- Bruk relevante verktøy når fakta må undersøkes.\n"
                "- Ikke finn på tall.\n"
                "- Ikke finn på kilder.\n"
                "- Ikke finn på SQL-resultater.\n"
                "- Bruk kun SELECT-spørringer mot databasen.\n"
                "- Skill mellom lokale data og eksterne kilder.\n"
                "- Forklar begrensninger i datagrunnlaget.\n"
                "- Oppgi kilder når verktøyresultatene "
                "inneholder kildeinformasjon."
            ),
        },
        {
            "role": "user",
            "content": user_prompt.strip(),
        },
    ]

    # --------------------------------------------------------
    # Agent-loop
    # --------------------------------------------------------

    for iteration in range(
        1,
        max_iterations + 1,
    ):

        try:

            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=messages,
                tools=TOOLS,
                tool_choice="auto",
            )

        except Exception as exc:

            return (
                f"Feil ved kall til modellen "
                f"'{MODEL_NAME}': "
                f"{type(exc).__name__}: {exc}"
            )

        # ----------------------------------------------------
        # Kontroller respons
        # ----------------------------------------------------

        if not response.choices:

            return (
                "Feil: Modellen returnerte "
                "ingen svaralternativer."
            )

        msg = response.choices[0].message

        # Lagre assistentens svar/verktøykall
        assistant_message = msg.model_dump(
            exclude_none=True
        )

        messages.append(
            assistant_message
        )

        # ----------------------------------------------------
        # Ingen tool calls -> ferdig svar
        # ----------------------------------------------------

        if not msg.tool_calls:

            return (
                msg.content
                or "Modellen returnerte et tomt svar."
            )

        # ----------------------------------------------------
        # Behandle tool calls
        # ----------------------------------------------------

        for tool_call in msg.tool_calls:

            func_name = (
                tool_call.function.name
            )

            raw_arguments = (
                tool_call.function.arguments
            )

            try:

                args = json.loads(
                    raw_arguments or "{}"
                )

            except json.JSONDecodeError as exc:

                tool_result = (
                    "Feil: Modellen returnerte "
                    "ugyldige JSON-argumenter: "
                    f"{exc}"
                )

            else:

                print(
                    "\n"
                    f"➜ [{MODEL_NAME} | "
                    f"iterasjon {iteration}]"
                )

                print(
                    f"Verktøy: {func_name}"
                )

                print(
                    f"Argumenter: {args}"
                )

                tool_result = execute_tool(
                    func_name=func_name,
                    args=args,
                )

                print(
                    f"Resultat:\n{tool_result}\n"
                )

            # ------------------------------------------------
            # Send tool-resultatet tilbake til modellen
            # ------------------------------------------------

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": str(tool_result),
                }
            )

    # --------------------------------------------------------
    # Maks antall iterasjoner
    # --------------------------------------------------------

    return (
        "Kunne ikke levere et endelig svar "
        f"innenfor {max_iterations} iterasjoner."
    )


# ============================================================
# LOKAL TEST
# ============================================================

if __name__ == "__main__":

    print()
    print("=" * 60)
    print("KRS KOMMUNE AI-AGENT")
    print("=" * 60)
    print(f"Modell: {MODEL_NAME}")
    print(f"Ollama: {USE_OLLAMA}")
    print("=" * 60)
    print()

    prompt = input(
        "Spør KRS-agenten: "
    ).strip()

    answer = run_krs_agent(
        prompt
    )

    print()
    print("=" * 60)
    print("SVAR")
    print("=" * 60)
    print()

    print(answer)


KRS KOMMUNE AI-AGENT
Modell: gemini-3.8-flash
Ollama: False


➜ [gemini-3.8-flash | iterasjon 1]
Verktøy: query_erp_transactions
Argumenter: {'query': 'SELECT substr(account_code, 1, 2) as art_gruppe, sum(amount) as total_amount FROM stg_erp_transactions WHERE amount > 0 GROUP BY art_gruppe ORDER BY total_amount DESC LIMIT 10;'}
Resultat:
[{"art_gruppe":"SK","total_amount":21316000.0},{"art_gruppe":"10","total_amount":12931477.7199999988},{"art_gruppe":"12","total_amount":458296.06},{"art_gruppe":"11","total_amount":160735.75},{"art_gruppe":"14","total_amount":35217.28},{"art_gruppe":"15","total_amount":9.27}]


➜ [gemini-3.8-flash | iterasjon 1]
Verktøy: search_knowledge_base
Argumenter: {'query': 'største kostnad utgifter lønn driftsutgifter rammeområde'}
Resultat:
Søkeresultat fra dokumenter for 'største kostnad utgifter lønn driftsutgifter rammeområde': [Fant avsnitt om rammer og tiltak].


➜ [gemini-3.8-flash | iterasjon 2]
Verktøy: query_erp_transactions
Argumenter: {'query': "S

In [23]:
import tools

print("Fil:", tools.__file__)
print("\nFunksjoner:")
print([x for x in dir(tools) if not x.startswith("_")])

Fil: c:\Users\frank\Desktop\MVP krs kommune\tools.py

Funksjoner:
['DB_PATH', 'ERPSQLQueryInput', 'KnowledgeSearchInput', 'pd', 'query_erp_transactions', 'search_knowledge_base', 'sqlite3']


In [27]:
import schemas

print([
    x
    for x in dir(schemas)
    if not x.startswith("_")
])

['BaseModel', 'ERPSQLQueryInput', 'Field', 'KnowledgeSearchInput', 'SerperSearchInput', 'importlib', 'schemas']


In [28]:
import importlib
import tools

importlib.reload(tools)

print([
    x
    for x in dir(tools)
    if not x.startswith("_")
])

['DB_PATH', 'ERPSQLQueryInput', 'KnowledgeSearchInput', 'SerperSearchInput', 'os', 'pd', 'query_erp_transactions', 'requests', 'search_knowledge_base', 'search_web_serper', 'sqlite3']


In [29]:
from tools import (
    query_erp_transactions,
    search_knowledge_base,
    search_web_serper,
)

print("✅ Alle 3 tools kan importeres")

✅ Alle 3 tools kan importeres


In [30]:
from agent import run_krs_agent

In [31]:
result = run_krs_agent(
    "Hva er de viktigste økonomiske "
    "utfordringene for Kristiansand kommune?"
)

print(result)

Feil ved kall til modellen 'gemini-2.0-flash': NotFoundError: Error code: 404 - [{'error': {'code': 404, 'message': 'This model models/gemini-2.0-flash is no longer available. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/get-started).', 'status': 'NOT_FOUND'}}]
